# 14 — Validated factual serving cutover

EVENT + LIVE remains LiveEventStateStore. EVENT post-session/replay can use the existing backend Database integration over Autonomous views. MODELLED remains Synthetic100.

Run 99 before publication; the same complete gate runs again here. Set `AIDP_EVENT_PUBLISH=true` only for deliberate Workbench publication, and provide `OCI_REGION`, `OCI_OBJECT_STORAGE_NAMESPACE`, `OCI_EVENT_ARCHIVE_BUCKET`, `OCI_EVENT_SERVING_PREFIX` matching the **existing** gold/serving volume LOCATION (normally `race_intelligence_serving/gold_event`). No catalog/volume is created. Versioned Parquet is read back before any Autonomous view cutover; existing datasets are retained.

Set `AIDP_EVENT_ADB_CUTOVER=true` with existing RACE_DB settings only during an approved maintenance window. External-table generation names prevent rewriting evidence; DBMS_CLOUD uses OCI$RESOURCE_PRINCIPAL. Oracle DDL commits per view, so complete this cell and its checks before enabling `EVENT_POST_SESSION_SOURCE=aidp`. No browser accesses AIDP. Older feeds are not exported by this notebook.


In [ ]:
# Resolve shared code from the existing notebook in this kernel.
import os
import sys
import json
from pathlib import Path
roots = [Path(os.environ["AIDP_PROJECT_ROOT"])] if os.getenv("AIDP_PROJECT_ROOT") else [Path.cwd(), *Path.cwd().parents]
root = next((p for p in roots if (p / "notebooks_aidp/10_event_capture_foundation.ipynb").is_file()), None)
if root is None:
    raise RuntimeError("Set AIDP_PROJECT_ROOT to the project checkout in Workbench.")
os.environ["AIDP_PROJECT_ROOT"] = str(root.resolve())
foundation = json.loads((root / "notebooks_aidp/10_event_capture_foundation.ipynb").read_text())
for cell in foundation["cells"]:
    if "aidp_shared" in cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(cell["source"]), "10_event_capture_foundation", "exec"), globals())
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Reusable read-only serving preflight; no runtime environment assignments.
import re
from urllib.parse import quote, urlsplit

SERVING_ENV = ("OCI_REGION", "OCI_OBJECT_STORAGE_NAMESPACE",
               "OCI_EVENT_ARCHIVE_BUCKET", "OCI_EVENT_SERVING_PREFIX")

def serving_config(environ):
    missing = [key for key in SERVING_ENV if not environ.get(key, "").strip()]
    require(not missing, "Missing serving configuration: " + ", ".join(missing) +
            ". Configure these variables explicitly in Workbench before publication.")
    region, namespace, bucket, raw_prefix = [environ[key] for key in SERVING_ENV]
    prefix = raw_prefix.strip("/")
    require(bool(re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)+", region)), "Invalid OCI region")
    require(bool(re.fullmatch(r"[A-Za-z0-9_-]+", namespace)), "Invalid OCI namespace")
    require(bucket == "example-event-archive", "Unexpected serving bucket")
    require(prefix and all(ch.isascii() and (ch.isalnum() or ch in "-_/") for ch in prefix)
            and all(prefix.split("/")), "Invalid serving prefix")
    return region, namespace, bucket, prefix

def volume_location(records):
    # Supported runtime shapes: wide location column or Attribute/Value rows.
    # Never infer a storage URI from the configured values.
    locations = []
    for row in records:
        values = {re.sub(r"[ _]", "", str(k).lower()): v for k, v in row.items()}
        for key in ("location", "storagelocation"):
            if key in values:
                locations.append(values[key])
        if str(values.get("attribute", "")).lower() in ("location", "storage location"):
            locations.append(values.get("value"))
    require(len(locations) == 1 and isinstance(locations[0], str) and bool(locations[0]),
            "Unsupported or ambiguous DESCRIBE VOLUME metadata. Inspect its column names and "
            "Location/storage_location representation in Workbench; update the explicit parser. "
            "No serving artifact was written.")
    return locations[0]

def verify_serving_volume(spark, root, config):
    region, namespace, bucket, prefix = config
    parts = root.split("/")
    require(len(parts) >= 6 and parts[:2] == ["", "Volumes"] and
            all(re.fullmatch(r"[A-Za-z0-9_]+", p) for p in parts[2:5]) and
            all(parts[5:]), "Invalid GOLD_EVENT_SERVING_ROOT")
    volume = ".".join(parts[2:5])
    try:
        rows = [r.asDict() for r in spark.sql("DESCRIBE VOLUME " + volume).collect()]
    except Exception:
        raise RuntimeError("Cannot inspect DESCRIBE VOLUME " + volume +
                           ". Check Workbench volume access and runtime support; no publication allowed.") from None
    location = volume_location(rows)
    parsed = urlsplit(location)
    require(parsed.scheme == "oci" and not parsed.query and not parsed.fragment and
            parsed.netloc == bucket + "@" + namespace,
            "Serving volume bucket/namespace does not match explicit OCI configuration")
    base = parsed.path.strip("/")
    suffix = "/".join(parts[5:])
    resolved = "/".join(filter(None, (base, suffix)))
    require(resolved == prefix, "Serving volume/root does not map to OCI_EVENT_SERVING_PREFIX")
    return {"volume": volume, "root": root, "mapping": "VERIFIED",
            "region": region, "prefix": prefix}

def require_publish(environ):
    require(environ.get("AIDP_EVENT_PUBLISH") == "true",
            "Validation passed. Set AIDP_EVENT_PUBLISH=true to deliberately publish to the existing serving volume.")

def require_adb_cutover(environ):
    require(environ.get("AIDP_EVENT_ADB_CUTOVER") == "true",
            "Parquet read-back verified; Autonomous NOT executed. Review the printed cutover plan, "
            "configure RACE_DB externally and set AIDP_EVENT_ADB_CUTOVER=true only in an approved maintenance window.")

def verify_oracle_schema(rows, contract):
    require([r[0].lower() for r in rows] == contract["column_names"],
            "Existing Autonomous generation column names/order are incompatible")
    for row, dtype in zip(rows, contract["spark_types"]):
        _, oracle_type, length, precision, scale = row
        if dtype == "string":
            ok = oracle_type == "VARCHAR2" and length == 4000
        elif dtype in ("bigint", "int", "boolean"):
            ok = oracle_type == "NUMBER" and precision == {"bigint": 19, "int": 10, "boolean": 1}[dtype] and scale == 0
        elif dtype == "double":
            ok = oracle_type == "BINARY_DOUBLE"
        elif dtype == "timestamp":
            ok = oracle_type == "TIMESTAMP(6)" and scale == 6
        else:
            ok = False
        require(ok, "Existing Autonomous generation type/size/precision is incompatible: " + contract["table_name"])


In [ ]:
# Validate the complete chain before producing any serving artifact.
validation = validate_all(spark)
require_publish(os.environ)
config = serving_config(os.environ)
region, namespace, bucket, prefix = config
print(verify_serving_volume(spark, GOLD_EVENT_SERVING_ROOT, config))
base_uri = f"https://objectstorage.{region}.oraclecloud.com/n/{quote(namespace, safe='')}/b/{quote(bucket, safe='')}/o/{prefix}"
# A generation is published fully before any view points at it. No existing data is deleted.
run_id = digest(packed(validation) + packed([
    r.content_hash for r in spark.table(table(BRONZE_SCHEMA, "event_segments")).orderBy("logical_session_id", "segment_id").select("content_hash").collect()
]) + "".join((PROJECT_ROOT / "notebooks_aidp" / name).read_text() for name in ["10_event_capture_foundation.ipynb", "11_event_capture_rehearsal.ipynb", "12_event_capture_silver.ipynb", "13_event_capture_gold.ipynb", "14_publish_event_serving.ipynb"]))[:12]
release_root = f"{GOLD_EVENT_SERVING_ROOT}/_releases/{run_id}"

cards = spark.table(table(GOLD_EVENT_SCHEMA, "baseball_card_evidence"))
summary = spark.table(table(GOLD_EVENT_SCHEMA, "session_summary"))
standings = spark.table(table(SILVER_SCHEMA, "event_standings")).select(
    F.col("logical_session_id").alias("session_id"), "source_epoch",
    F.concat_ws(":", "segment_id", F.col("event_sequence")).alias("snapshot_id"),
    F.concat_ws(":", "segment_id", F.col("event_sequence")).alias("source_snapshot_key"),
    "participant_id", F.col("kart_number").alias("external_participant_number"),
    "observed_at", "recorded_at", "position", "lap_number",
    F.col("gap_to_leader").alias("gap_first_seconds"), F.col("interval").alias("gap_previous_seconds"),
    F.lit("AKS").alias("source"), "operational_context", "evidence_domain", "event_contract_version")
positions = spark.table(table(SILVER_SCHEMA, "event_positions")).select(
    F.col("logical_session_id").alias("session_id"), "source_epoch",
    F.concat_ws(":", "segment_id", F.col("event_sequence"), F.col("row_index")).alias("sample_id"),
    "source_entity_id", "participant_id", F.col("kart_number").alias("external_participant_number"),
    "observed_at", "recorded_at", "position_source", "latitude", "longitude",
    F.lit("AKS").alias("source"), "operational_context", "evidence_domain", "event_contract_version")
# Preserve the existing replay SQL/API column contract, with absent observations NULL.
replay_optional = {
    "source_timestamp": "timestamp", "ingested_at": "timestamp",
    "elapsed_time_seconds": "double", "gap_first_laps": "bigint", "gap_previous_laps": "bigint",
    "status": "string", "position_change": "bigint", "at_loop": "string",
}
for field, dtype in replay_optional.items():
    if field not in standings.columns:
        standings = standings.withColumn(field, F.lit(None).cast(dtype))
for field in ["source_timestamp", "ingested_at"]:
    positions = positions.withColumn(field, F.lit(None).cast("timestamp"))
for field in ["raw_status_bits", "unknown_status_bits", "is_official_car", "is_leader", "is_lapped",
              "is_class_leader", "is_stopped", "is_class_lapped", "is_in_pit", "is_out_lap",
              "is_position_unknown", "is_retired", "is_late", "is_entry_list_target", "is_no_gps", "is_chequered"]:
    positions = positions.withColumn(field, F.lit(None).cast("bigint"))
products = {"baseball_card_evidence": cards, "session_summary": summary,
            "standing_replay_feed": standings, "track_replay_feed": positions}
# Snapshot-only data products; no old full card feed, display names or modelled fields are exported.
oracle_types = {"string": "VARCHAR2(4000)", "bigint": "NUMBER(19)", "int": "NUMBER(10)",
                "double": "BINARY_DOUBLE", "timestamp": "TIMESTAMP(6)", "boolean": "NUMBER(1)"}
contracts = {}
for name, df in products.items():
    assert_private_free(df.columns)
    target = f"{release_root}/{name}"
    path = Path(target)
    if path.exists():
        restored = spark.read.parquet(target).select(df.columns)
        require(not df.exceptAll(restored).limit(1).count() and not restored.exceptAll(df).limit(1).count(),
                "CONFLICT: existing serving generation differs; use a new validated generation")
    else:
        df.write.mode("errorifexists").parquet(target)
    restored = spark.read.parquet(target).select(df.columns)
    require(not df.exceptAll(restored).limit(1).count() and not restored.exceptAll(df).limit(1).count(),
            "Serving read-back mismatch")
    assert_private_free(restored.columns)
    contracts[name] = {
        "table_name": ("DE_" + {"baseball_card_evidence":"CARD","session_summary":"SESSION",
                        "standing_replay_feed":"STANDING","track_replay_feed":"TRACK"}[name] + "_" + run_id).upper(),
        "file_uri": f"{base_uri}/_releases/{run_id}/{name}/*.parquet",
        "columns": ", ".join(f'"{f.name.upper()}" {oracle_types[f.dataType.simpleString()]}' for f in df.schema),
        "column_names": df.columns,
        "spark_types": [f.dataType.simpleString() for f in df.schema],
        "expected_rows": df.count(),
    }

card_ext = contracts["baseball_card_evidence"]["table_name"]
summary_ext = contracts["session_summary"]["table_name"]
standing_ext = contracts["standing_replay_feed"]["table_name"]
track_ext = contracts["track_replay_feed"]["table_name"]
latest = f"(SELECT logical_session_id, MAX(source_epoch) source_epoch FROM {summary_ext} GROUP BY logical_session_id)"
views = {
    "V_EVENT_BASEBALL_CARD_EVIDENCE": f"SELECT c.* FROM {card_ext} c JOIN {latest} e ON c.logical_session_id=e.logical_session_id AND c.source_epoch=e.source_epoch",
    "V_EVENT_RACE_SUMMARY": f"""SELECT s.logical_session_id session_id, s.logical_session_id event_id,
        'RACE' session_type, s.session_status status, s.started_at, s.ended_at, s.participant_count,
        'AKS' source, s.operational_context, s.evidence_domain factual_evidence_domain,
        s.event_contract_version, s.source_epoch FROM {summary_ext} s JOIN {latest} e
        ON s.logical_session_id=e.logical_session_id AND s.source_epoch=e.source_epoch""",
    "V_EVENT_PARTICIPANT_CARD_FEED": """SELECT logical_session_id session_id, participant_id,
        kart_number external_participant_number, kart_number kart_id, 'CHALLENGER' participant_type,
        start_position actual_start_position, peak_observed_position, latest_observed_position,
        final_position actual_final_position, position_change, best_valid_lap actual_best_lap_seconds,
        CAST(NULL AS NUMBER) actual_completed_laps, observed_lap, 'AKS' data_provenance, 'PARTIAL' card_data_status,
        operational_context, evidence_domain, event_contract_version, source_epoch
        FROM V_EVENT_BASEBALL_CARD_EVIDENCE""",
    "V_EVENT_PARTICIPANT_RACE_SUMMARY": """SELECT logical_session_id session_id, participant_id,
        kart_number external_participant_number, kart_number kart_id, 'CHALLENGER' participant_type,
        start_position initial_position, final_position, position_change, best_valid_lap best_lap_seconds,
        CAST(NULL AS NUMBER) lap_count, observed_lap, latest_observed_position, peak_observed_position, updated_at last_observed_at,
        'AKS' source, operational_context, evidence_domain, event_contract_version, source_epoch
        FROM V_EVENT_BASEBALL_CARD_EVIDENCE""",
    "V_EVENT_STANDING_REPLAY": f"SELECT r.* FROM {standing_ext} r JOIN {latest} e ON r.session_id=e.logical_session_id AND r.source_epoch=e.source_epoch",
    "V_EVENT_TRACK_REPLAY": f"SELECT r.* FROM {track_ext} r JOIN {latest} e ON r.session_id=e.logical_session_id AND r.source_epoch=e.source_epoch",
}
# The plan contains only object names, locations and counts, never row payloads.
print({"autonomous_cutover_plan": {"run_id": run_id,
    "products": {name: {"external_table": c["table_name"], "parquet_uri": c["file_uri"],
                        "expected_rows": c["expected_rows"]} for name, c in contracts.items()},
    "views_create_or_replace": list(views),
    "backend_source": "UNCHANGED"}})
require_adb_cutover(os.environ)
# No Database construction, connection or Autonomous DDL occurs before this gate.
from app.config import Settings
from app.db import Database
database = Database(Settings.from_env())
with database.connection() as connection:
    with connection.cursor() as cursor:
        schema_query = ("SELECT column_name, data_type, data_length, data_precision, data_scale "
                        "FROM user_tab_columns WHERE table_name=:name ORDER BY column_id")
        existing_tables = set()
        # Inspect every reused generation before creating any object or changing a view.
        for contract in contracts.values():
            cursor.execute(schema_query, {"name": contract["table_name"]})
            existing = cursor.fetchall()
            if existing:
                verify_oracle_schema(existing, contract)
                cursor.execute("SELECT COUNT(*) FROM user_external_tables WHERE table_name=:name",
                               {"name": contract["table_name"]})
                require(cursor.fetchone()[0] == 1, "Existing generated object is not an external table")
                existing_tables.add(contract["table_name"])
        for contract in contracts.values():
            if contract["table_name"] not in existing_tables:
                cursor.execute("""BEGIN DBMS_CLOUD.CREATE_EXTERNAL_TABLE(
                    table_name => :name, credential_name => 'OCI$RESOURCE_PRINCIPAL',
                    file_uri_list => :uri, column_list => :cols,
                    format => JSON_OBJECT('type' VALUE 'parquet')); END;""",
                    {"name": contract["table_name"], "uri": contract["file_uri"], "cols": contract["columns"]})
            cursor.execute(schema_query, {"name": contract["table_name"]})
            verify_oracle_schema(cursor.fetchall(), contract)
            cursor.execute("SELECT COUNT(*) FROM " + contract["table_name"])
            require(cursor.fetchone()[0] == contract["expected_rows"],
                    "Autonomous URI/row-count mismatch")
        # Compare the actual Autonomous card values, not just row counts.
        cursor.execute("SELECT " + ", ".join('"' + c.upper() + '"' for c in cards.columns) + " FROM " + card_ext)
        remote_cards = [dict(zip(cards.columns, row)) for row in cursor.fetchall()]
        def comparable_card(row):
            result = dict(row)
            for field in cards.schema:
                value = result[field.name]
                if value is None:
                    continue
                dtype = field.dataType.simpleString()
                if dtype == "timestamp":
                    result[field.name] = stamp(value)
                elif dtype in {"bigint", "int"}:
                    result[field.name] = int(value)
                elif dtype == "double":
                    result[field.name] = float(value)
            return packed(result)
        require(sorted(comparable_card(r) for r in remote_cards) ==
                sorted(comparable_card(r.asDict()) for r in cards.toLocalIterator()),
                "Autonomous factual card metric read-back mismatch")
        # Perform deliberately in a maintenance window: Oracle DDL commits per view.
        for name, query in views.items():
            cursor.execute("CREATE OR REPLACE VIEW " + name + " AS " + query)
        for view_name in views:
            cursor.execute("SELECT * FROM " + view_name + " WHERE 1=0")
            assert_private_free([c[0] for c in cursor.description])
print({"serving": "VERIFIED", "generation": run_id, "live_source": "UNCHANGED",
       "backend_activation": "Set EVENT_POST_SESSION_SOURCE=aidp after read-back verification"})
